# 02 — EDA: Acidentes Rodoviários PRF (2017-2025)

Responde às perguntas do TCC com dados atualizados, comparando o **triênio original (2017-2020, pré-pandemia)** ao **quadriênio novo (2021-2025, pós-pandemia)**.

**Pré-requisito:** ter rodado `make data && make clean`. Para a comparação histórica, copie também os CSVs originais do TCC (`datatran2017-2020.csv`) para `data/raw/` antes do `make clean`.

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from src.features.build import build_features, load_processed

df = build_features(load_processed())
df['periodo'] = np.where(df['ano'] <= 2020, 'pre_pandemia (2017-2020)', 'pos_pandemia (2021-2025)')
print(f'Linhas: {len(df):,}  |  Anos: {sorted(df["ano"].dropna().unique())}')
df.head(3)

## 1. Volume e fatalidades por ano
Resposta para: *Qual a quantidade de ocorrências e fatalidade ao longo dos anos?*

In [ ]:
por_ano = (
    df.groupby('ano').agg(
        acidentes=('data', 'size'),
        mortos=('mortos', 'sum'),
        feridos_leves=('feridos_leves', 'sum'),
        feridos_graves=('feridos_graves', 'sum'),
    ).reset_index()
)
por_ano['letalidade_pct'] = (por_ano['mortos'] / por_ano['acidentes'] * 100).round(2)
por_ano

In [ ]:
fig = px.bar(por_ano, x='ano', y='acidentes', text='acidentes', title='Acidentes por ano')
fig.show()
fig = px.line(por_ano, x='ano', y='letalidade_pct', markers=True, title='Letalidade (% mortos / acidentes)')
fig.show()

### Observação esperada
A pandemia (2020) deve mostrar **queda abrupta** no volume de acidentes (menos circulação). Já a **letalidade** tende a **subir** — quem rodou, rodou em estradas vazias e mais rápido. Confirme isso nos dados acima.

## 2. Principais causas e tipos
Resposta para: *Quais as principais causas de acidentes e as causas mais letais? E os tipos?*

In [ ]:
top_causas = (
    df.groupby('causa_acidente')
      .agg(acidentes=('data', 'size'), mortos=('mortos', 'sum'))
      .assign(letalidade_pct=lambda x: x['mortos'] / x['acidentes'] * 100)
      .sort_values('acidentes', ascending=False)
      .head(20)
      .reset_index()
)
px.bar(top_causas, x='acidentes', y='causa_acidente', orientation='h', title='Top 20 causas (volume)').show()
px.bar(top_causas.sort_values('letalidade_pct', ascending=False), x='letalidade_pct', y='causa_acidente', orientation='h', title='Top 20 causas (letalidade %)').show()

In [ ]:
top_tipos = (
    df.groupby('tipo_acidente')
      .agg(acidentes=('data', 'size'), mortos=('mortos', 'sum'))
      .assign(letalidade_pct=lambda x: x['mortos'] / x['acidentes'] * 100)
      .sort_values('mortos', ascending=False)
      .head(15)
      .reset_index()
)
px.bar(top_tipos, x='mortos', y='tipo_acidente', orientation='h', title='Tipos de acidente mais letais').show()

## 3. Sazonalidade
Resposta para: *Quais dias e horários são mais perigosos?*

In [ ]:
dias_ordem = ['segunda-feira','terça-feira','quarta-feira','quinta-feira','sexta-feira','sábado','domingo']
por_dia = df.groupby('dia_semana').agg(acidentes=('data', 'size'), mortos=('mortos', 'sum')).reset_index()
por_dia = por_dia.set_index('dia_semana').reindex(dias_ordem).reset_index()
px.bar(por_dia, x='dia_semana', y=['acidentes', 'mortos'], barmode='group', title='Acidentes e mortes por dia da semana').show()

In [ ]:
por_hora = df.groupby('hora').agg(acidentes=('data', 'size'), mortos=('mortos', 'sum')).reset_index()
fig = go.Figure()
fig.add_bar(x=por_hora['hora'], y=por_hora['acidentes'], name='Acidentes')
fig.add_scatter(x=por_hora['hora'], y=por_hora['mortos'], mode='lines+markers', name='Mortos', yaxis='y2')
fig.update_layout(yaxis2=dict(overlaying='y', side='right'), title='Acidentes e mortos por hora do dia')
fig.show()

## 4. Geografia
Resposta para: *Quais BRs, estados e regiões têm mais acidentes / são mais letais?*

In [ ]:
por_br = (
    df.groupby('br')
      .agg(acidentes=('data', 'size'), mortos=('mortos', 'sum'))
      .sort_values('mortos', ascending=False)
      .head(20)
      .reset_index()
)
px.bar(por_br, x='mortos', y='br', orientation='h', title='Top 20 BRs por mortes').show()

In [ ]:
por_uf = (
    df.groupby('uf')
      .agg(acidentes=('data', 'size'), mortos=('mortos', 'sum'))
      .assign(letalidade_pct=lambda x: x['mortos'] / x['acidentes'] * 100)
      .sort_values('acidentes', ascending=False)
      .reset_index()
)
px.bar(por_uf, x='uf', y='acidentes', title='Acidentes por UF', color='letalidade_pct', color_continuous_scale='reds').show()

## 5. Condições meteorológicas
Resposta para: *Quais condições meteorológicas mais influenciam?*

In [ ]:
por_clima = (
    df.groupby('condicao_metereologica')
      .agg(acidentes=('data', 'size'), mortos=('mortos', 'sum'))
      .assign(letalidade_pct=lambda x: x['mortos'] / x['acidentes'] * 100)
      .sort_values('acidentes', ascending=False)
      .reset_index()
)
por_clima

## 6. Comparativo pré vs pós-pandemia
Pergunta nova do v2: *Como mudou o perfil dos acidentes entre 2017-2020 e 2021-2025?*

In [ ]:
comp = (
    df.groupby('periodo')
      .agg(
          acidentes=('data', 'size'),
          mortos=('mortos', 'sum'),
          feridos=('feridos_leves', 'sum'),
      )
      .assign(
          letalidade_pct=lambda x: (x['mortos'] / x['acidentes'] * 100).round(2),
          mortos_por_100=lambda x: (x['mortos'] / x['acidentes'] * 100).round(2),
      )
)
comp

In [ ]:
# Distribuição da gravidade entre os períodos
tab = pd.crosstab(df['periodo'], df['gravidade'], normalize='index').mul(100).round(2)
tab

In [ ]:
# Top 10 causas que mais MUDARAM entre os períodos (variação relativa)
causa_periodo = df.groupby(['periodo', 'causa_acidente']).size().unstack(level=0, fill_value=0)
if causa_periodo.shape[1] == 2:
    causa_periodo.columns = ['pos', 'pre'] if causa_periodo.columns[0].startswith('pos') else ['pre', 'pos']
    causa_periodo['var_pct'] = ((causa_periodo['pos'] - causa_periodo['pre']) / causa_periodo['pre'].replace(0, np.nan) * 100)
    causa_periodo = causa_periodo[(causa_periodo['pre'] >= 500) | (causa_periodo['pos'] >= 500)]
    print('Causas que mais cresceram:')
    display(causa_periodo.sort_values('var_pct', ascending=False).head(10))
    print('\nCausas que mais caíram:')
    display(causa_periodo.sort_values('var_pct').head(10))
else:
    print('Dados de apenas um período disponíveis.')

## 7. Insights consolidados

Use as células acima para responder, no relatório final, perguntas como:

- O **número absoluto** de acidentes caiu pós-pandemia? E a **letalidade**?
- A distribuição de **dias da semana** e **horários** mudou (ex.: mais acidentes em horário comercial pós-home-office)?
- Quais **causas humanas** (álcool, celular, sono) crescem ou diminuem?
- A concentração geográfica (top BRs / UFs) permanece a mesma?

Anote os achados aqui para virarem narrativa no dashboard e no artigo de portfólio.